# Full-descent ravine — debug notebook

Runs `GradientDescent + RavineStep + BacktrackingLineSearch` and plots the geometry with explicit labels:

* $v^i$ — extrapolation points (cyan diamonds).
* $x^i$ — **converged bottoms** at the end of each inner descent (bright red, large). The next ravine extrapolation is built from two consecutive $x^i$.
* $d^i_j$ — intermediate gradient iterates *during* the inner descent from $v^i$ towards $x^i$ (faded pink, small).

**Why:** we want to see whether the inner descents actually reach the ravine bottom, and whether the resulting $x^{i-1} \to x^i$ direction then aligns with the valley so the ravine extrapolation is meaningful.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import numpy as np
import plotly.graph_objects as go
from engine.models import TargetFunction
from engine.optimizers.gradient_methods import GradientDescent
from engine.strategies.gd_step import RavineStep, VanillaGradientStep
from engine.strategies.step_size import BacktrackingLineSearch, FixedStepSize
from engine.strategies.stopping import MaxGenerationsCriterion

logging.getLogger('engine.utils').setLevel(logging.WARNING)

EXPR = '100*x**2 + y**2'
BOUNDS = [(-5, 5), (-5, 5)]
START = np.array([4.0, 4.0])
MAX_OUTER_ITERS = 10
INNER_TOL = 0.3         # set to None to stop only by INNER_MAX_ITER
INNER_MAX_ITER = 30

target = TargetFunction(EXPR, bounds=BOUNDS)
opt = GradientDescent(
    target,
    start_pos=START,
    step_strategy=RavineStep(
        inner_strategy=VanillaGradientStep(),
        ravine_step_size_strategy=FixedStepSize(0.5),
        ravine_shift=0.5,
        inner_tol=INNER_TOL,
        inner_max_iter=INNER_MAX_ITER,
    ),
    step_size_strategy=BacktrackingLineSearch(),    # used by the inner VanillaGradientStep
    stopping_criterion=MaxGenerationsCriterion(),
)
results = opt.run(max_iter=MAX_OUTER_ITERS)
print(f'Outer iterations run: {results.iterations}')
print(f'Final x: {results.final_population[0]}, f(x) = {results.final_f:.6f}')
print(f'History frames: {len(results.history)}')

Outer iterations run: 16
Final x: [-0.08018077  4.03029227], f(x) = 16.886151
History frames: 16


## Walk the history and assign labels

`RavineStep` emits these frames per outer iteration:

* **Bootstrap (iter 1).** `Optimizer.run()` first appends an *untagged* frame — that is $v^0$. Then `step()` runs the inner descent from $v^0$ (a sequence of `descend` frames), then `extrapolate`($v^1$), then another inner descent (`descend` frames) from $v^1$.
* **Subsequent iters.** `run()` appends an *untagged* frame which duplicates the previous $x^i$, then `step()` appends `extrapolate`($v^{i+1}$) followed by another descent block ending at $x^{i+1}$.

Labeling rule: in each consecutive `descend` block the **last** frame is $x^i$ (next ravine extrapolation builds from it); the rest are $d^i_j$, where $i$ matches the $v^i$ being descended from. Untagged duplicates after the first are skipped.

In [3]:
SUP = '⁰¹²³⁴⁵⁶⁷⁸⁹'

def sup(k):
    return SUP[k] if 0 <= k < 10 else f'^{k}'

def label_history(history):
    """Return list of (point, label, kind) where kind in {'v','x','d'}."""
    labeled = []
    seen_v0 = False
    current_i = 0   # which v^i the current descent block is descending from
    j = 0           # inner-step counter within the current block
    n = len(history)
    for idx, frame in enumerate(history):
        phase = frame.get('phase')
        pt = frame['population'][0]
        if phase is None:
            if not seen_v0:
                labeled.append((pt, f'v{sup(0)}', 'v'))
                seen_v0 = True
            # else: duplicate of previous x^i — skip
        elif phase == 'extrapolate':
            current_i += 1
            j = 0
            labeled.append((pt, f'v{sup(current_i)}', 'v'))
        elif phase == 'descend':
            is_last_in_block = (idx == n - 1) or (history[idx + 1].get('phase') != 'descend')
            if is_last_in_block:
                labeled.append((pt, f'x{sup(current_i)}', 'x'))
                j = 0
            else:
                j += 1
                labeled.append((pt, f'd{sup(current_i)}_{j}', 'd'))
    return labeled

labeled = label_history(results.history)
v_pts = [(p, l) for p, l, k in labeled if k == 'v']
x_pts = [(p, l) for p, l, k in labeled if k == 'x']
d_pts = [(p, l) for p, l, k in labeled if k == 'd']

print(f'{len(labeled)} labeled points: {len(v_pts)} v, {len(x_pts)} x, {len(d_pts)} d.\n')

# Per-block summary: v^i -> #inner steps -> x^i.
print(f'  {"start":>5}  {"vx":>9}  {"vy":>9}     {"#inner":>6}     {"end":>5}  {"xx":>9}  {"xy":>9}  {"f(xⁱ)":>10}')
print(f'  {"-----":>5}  {"---------":>9}  {"---------":>9}     {"------":>6}     {"-----":>5}  {"---------":>9}  {"---------":>9}  {"----------":>10}')
for i in range(len(v_pts)):
    v_pt, v_lbl = v_pts[i]
    n_inner = sum(1 for _, l, _ in labeled if l.startswith(f'd{sup(i)}_'))
    if i < len(x_pts):
        x_pt, x_lbl = x_pts[i]
        print(f'  {v_lbl:>5}  {v_pt[0]:>+9.4f}  {v_pt[1]:>+9.4f}     {n_inner:>6}     {x_lbl:>5}  {x_pt[0]:>+9.4f}  {x_pt[1]:>+9.4f}  {target.evaluate(x_pt):>10.4f}')
    else:
        print(f'  {v_lbl:>5}  {v_pt[0]:>+9.4f}  {v_pt[1]:>+9.4f}     {n_inner:>6}     (descent block did not produce a final x)')

16 labeled points: 2 v, 2 x, 12 d.

  start         vx         vy     #inner       end         xx         xy       f(xⁱ)
  -----  ---------  ---------     ------     -----  ---------  ---------  ----------
     v⁰    +4.0000    +4.0000          6        x⁰    -0.0713    +3.5825     13.3421
     v¹    +4.5000    +4.5000          6        x¹    -0.0802    +4.0303     16.8862


In [4]:
# Contour landscape (matches the UI's setup).
x_range = np.linspace(BOUNDS[0][0], BOUNDS[0][1], 300)
y_range = np.linspace(BOUNDS[1][0], BOUNDS[1][1], 300)
X, Y = np.meshgrid(x_range, y_range)
Z = target.evaluate([X, Y])

fig = go.Figure(data=[go.Contour(x=x_range, y=y_range, z=Z, colorscale='Viridis')])

# Faint chronological connector through every labeled point.
all_xy = np.array([p for p, _, _ in labeled])
fig.add_trace(go.Scatter(
    x=all_xy[:, 0], y=all_xy[:, 1], mode='lines',
    line=dict(color='rgba(255,255,255,0.35)', width=1, dash='dot'),
    name='visit order', hoverinfo='skip',
))

# d^i_j intermediate descent points — small, faded, small labels.
if d_pts:
    fig.add_trace(go.Scatter(
        x=[p[0] for p, _ in d_pts], y=[p[1] for p, _ in d_pts],
        mode='markers+text',
        marker=dict(color='rgba(255,170,170,0.55)', size=6, symbol='circle'),
        text=[l for _, l in d_pts],
        textfont=dict(color='rgba(255,200,200,0.75)', size=10),
        textposition='top center',
        name='dⁱⱼ (inner descent)',
    ))

# v^i extrapolation points — cyan diamonds.
if v_pts:
    fig.add_trace(go.Scatter(
        x=[p[0] for p, _ in v_pts], y=[p[1] for p, _ in v_pts],
        mode='markers+text',
        marker=dict(color='cyan', size=14, symbol='diamond', line=dict(color='black', width=1.5)),
        text=[l for _, l in v_pts],
        textfont=dict(color='cyan', size=16),
        textposition='top right',
        name='vⁱ (extrapolation)',
    ))

# x^i converged bottoms — bright, large, drawn on top.
if x_pts:
    fig.add_trace(go.Scatter(
        x=[p[0] for p, _ in x_pts], y=[p[1] for p, _ in x_pts],
        mode='markers+text',
        marker=dict(color='#ff2222', size=16, symbol='circle',
                    line=dict(color='white', width=2)),
        text=[l for _, l in x_pts],
        textfont=dict(color='#ff5555', size=18),
        textposition='bottom left',
        name='xⁱ (converged bottom — next ravine step from here)',
    ))

fig.update_layout(
    title=f'RavineStep + BacktrackingLineSearch on {EXPR} from {tuple(START)}',
    xaxis_title='x', yaxis_title='y',
    width=900, height=780,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01,
                bgcolor='rgba(0,0,0,0.55)', font=dict(color='white')),
)
fig.show()